# HW1 - Intro to the Map Reduce Paradigm  

__`MIDS w261: Machine Learning at Scale | UC Berkeley School of Information`__

Welcome to Machine Learning at Scale! This first homework assignment introduces one of the core strategies in distributed processing: divide and conquer. We'll use the simplest of tasks, word counting, to illustrate the difference between a scalable and non-scalable algorithm. You will be working with the text of _Alice in Wonderland_ to put these ideas into practice using Python and Bash scripting. By the end of this week you should be able to:
* ... __describe__ the Bias-Variance tradeoff as it applies to Machine Learning.
* ... __explain__ why we consider word counting to be an "Embarrassingly Parallel" task.
* ... __estimate__ the runtime of embarrassingly parallel tasks using "back of the envelope" calculations.
* ... __implement__ a Map Reduce algorithm using the Command Line.
* ... __set-up__ a Docker container and know why we use them for this course.

You will also  become familiar (if you aren't already) with `defaultdict`, `re` and `time` in Python, linux piping and sorting, and Jupyter magic commands `%%writefile` and `%%timeit`. 


__IMPORTANT:__ If you're not familiar with linux, you should read the following tutorial reagrding **piping** and **redirecting**: https://ryanstutorials.net/linuxtutorial/piping.php You will need to understand the differences to answer some of the later questions.

# Notebook Set-Up
Before starting your homework run the following cells to confirm your setup.

In [1]:
# Change the working directory to your local media notebook location 
%cd /media/notebooks/Assignments/HW1

/media/notebooks/Assignments/HW1


In [2]:
# Confirm you are running Python 3:
import sys
sys.version_info

sys.version_info(major=3, minor=8, micro=15, releaselevel='final', serial=0)

In [3]:
# Imports:
import re
import sys

Create a folder for any data you download locally.

In [4]:
!mkdir data
# NOTE: the contents of this directory will be ignored by git.

mkdir: cannot create directory ‘data’: File exists


In [5]:
# Install dos2unix just in case.
!sudo apt install dos2unix

Reading package lists... Done
Building dependency tree       
Reading state information... Done
The following NEW packages will be installed:
  dos2unix
0 upgraded, 1 newly installed, 0 to remove and 5 not upgraded.
Need to get 391 kB of archives.
After this operation, 1339 kB of additional disk space will be used.
Get:1 https://archive.debian.org/debian buster/main amd64 dos2unix amd64 7.4.0-1 [391 kB]
Fetched 391 kB in 0s (3085 kB/s)

78Selecting previously unselected package dos2unix.
(Reading database ... 185489 files and directories currently installed.)
Preparing to unpack .../dos2unix_7.4.0-1_amd64.deb ...
7Progress: [  0%] [..........................................................] 87Progress: [ 20%] [###########...............................................] 8Unpacking dos2unix (7.4.0-1) ...
7Progress: [ 40%] [#######################...................................] 8Setting up dos2unix (7.4.0-1) ...
7Progress: [ 60%] [##################################..........

# Question 1: Introductions

`The Caterpillar and Alice looked at each other for some time in silence: at last the Caterpillar took the hookah out of its mouth, and addressed her in a languid, sleepy voice. "Who are you?" said the Caterpillar.`

<div style="text-align: right"> -- Lewis Carroll, <i>Alice's Adventures in Wonderland</i>, Chapter 4 </div>


__a) Short Essay:__ Tell us about yourself! Briefly describe where you live, how far along you are in MIDS, what other classes you are taking and what you want to get out of w261.

In [6]:
# q1
### SHORT RESPONSE
### QUESTION: Tell us about yourself! Briefly describe where you live, how far along you are in MIDS,
#             what other classes you are taking and what you want to get out of w261.

### ENTER ANSWER IN BETWEEN THE """ """ INSIDE THE PRINT STATEMENT.

print(
"""
My name is Leah Finger and I live in Minneapolis, Minnesota (Midwest strong!). I don't have a strong background in computer
science - I was an economics undergrad major but I taught myself the basics of data science in and outside of my first job 
because it was fascinating to me. I took all the foundational courses plus 241 - Experiments for Causal Inference I've 
pretty much loved every class in MIDS so far. I am really looking to get more comfortable with cloud computing in this 
class - I am not very comfortable working out of VMs and such and I really want to get better and more confident doing that.
I also very much want to improve my algorithmic thinking and am excited (also a little intimidated) about the "you want it 
you build it" approach to model building in this class!
"""
)


My name is Leah Finger and I live in Minneapolis, Minnesota (Midwest strong!). I don't have a strong background in computer
science - I was an economics undergrad major but I taught myself the basics of data science in and outside of my first job 
because it was fascinating to me. I took all the foundational courses plus 241 - Experiments for Causal Inference I've 
pretty much loved every class in MIDS so far. I am really looking to get more comfortable with cloud computing in this 
class - I am not very comfortable working out of VMs and such and I really want to get better and more confident doing that.
I also very much want to improve my algorithmic thinking and am excited (also a little intimidated) about the "you want it 
you build it" approach to model building in this class!



# Question 2: Bias - Variance
__a) Short Essay:__ In 1-2 sentences (~200 and absolutely no more than 300 words!), explain the bias-variance trade off. Describe what it means to "decompose" sources of error. How is this used in machine learning? Please also cite any sources that informed your answer.

In [7]:
# q2
### SHORT RESPONSE
### QUESTION: In 1-2 sentences (~200 and absolutely no more than 300 words!), explain the bias-variance trade off.
#             Describe what it means to "decompose" sources of error. How is this used in machine learning?
#             Please also cite any sources that informed your answer.

### ENTER ANSWER IN BETWEEN THE """ """ INSIDE THE PRINT STATEMENT.

print(
"""
The bias variance tradeoff, according to our class discussion and reading from the ISLP textbook, is a tradeoff 
between high model flexibility (variance) and high model interpretability (bias); if the bias-variance tradeoff 
is not optimal, it leads to the problem of overfitting: too much variance, a model that is too complex and too 
well fit to the data, leads to overfitting which causes poor test set or real-world generalizability 
performance, so you need a balance of simplicity or bias and variance or fit to the data in your model. 
Error decomposition means decomposing error into reducable error that is suitable for your specific problem, 
such as MSE, into bias and variance via bootstrap analysis in a simulated world where we know the true "f" 
function (target function) if possible, and then irreducible error, such as measurement error in data collection process 
or just "noise" in data (note- if true f not knowable, assume 0 noise or irreducable error).
"""
)


The bias variance tradeoff, according to our class discussion and reading from the ISLP textbook, is a tradeoff 
between high model flexibility (variance) and high model interpretability (bias); if the bias-variance tradeoff 
is not optimal, it leads to the problem of overfitting: too much variance, a model that is too complex and too 
well fit to the data, leads to overfitting which causes poor test set or real-world generalizability 
performance, so you need a balance of simplicity or bias and variance or fit to the data in your model. 
Error decomposition means decomposing error into reducable error that is suitable for your specific problem, 
such as MSE, into bias and variance via bootstrap analysis in a simulated world where we know the true "f" 
function (target function) if possible, and then irreducible error, such as measurement error in data collection process 
or just "noise" in data (note- if true f not knowable, assume 0 noise or irreducable error).



# Question 3: Tokenizing
A number of our assignments this term will involve extracting information from text. A common preprocessing step when working with raw files is to 'tokenize' (i.e. extract words from) the text. Within the field of Natural Language Processing a lot of thought goes into what specific tokenizing makes most sense for a given task. For example, you might choose to remove punctuation or to consider punctuation symbols  'tokens' in their own right. __In this question you'll use the Python `re` module to create a tokenizer to use when you perform Word Count on the _Alice In Wonderland_ text.__

### Q3 Tasks:

* __a) Short Essay:__ In the Naive Bayes algorithm (which we'll implement next week), we'll estimate the _likelihood_ of a word by counting the number of times it appears and dividing by the size of the vocabulary (total number of unique words). Using the text: *"Alice had an adventure that took alice to wonderland"*, give a concrete example of how two different tokenizers could cause us to get two different results on this calculation. [`HINT`: _you should not need to read up on Naive Bayes to answer this question._]  
  

* __b) Multiple Choice:__ When tokenizing in this assignment we'll remove punctuation and discard numerical digits by making everything lowercase and then capturing only consecutive letters a to z. Suppose __`tokenize(x)`__ is a Python function that performs the desired tokenization. What would __`tokenize("By-the-bye, what became of Alice's 12 hats?!")`__ output?


* __c) code:__  Fill in the regular expression pattern in the cell labeled `part c` so that the subsequent call to `re.findall(RE_PATTERN, ...)` returns the tokenization described above. [`HINT`: _we've taken care of the lowercase part for you. If regex is new to you, check out the [`re`  documentation](https://docs.python.org/3/library/re.html) or [this PyMOTW tutorial](https://pymotw.com/2/re/)._]

### Q3 Student Answers:

> __a)__ Type your answer below!

In [8]:
# q3a
### SHORT RESPONSE
### QUESTION: In the Naive Bayes algorithm (which we'll implement next week), we'll estimate the _likelihood_ of a word
#       by counting the number of times it appears and dividing by the size of the vocabulary (total number of unique words).
#       Using the text: *"Alice had an adventure that took alice to wonderland"*, give a concrete example of how two
#       different tokenizers could cause us to get two different results on this calculation.
#   [`HINT`: _you should not need to read up on Naive Bayes to answer this question._]
# 
### ENTER ANSWER IN BETWEEN THE """ """ INSIDE THE PRINT STATEMENT.

print(
"""
A concrete example of two tokenizers giving different results on word count is if one tokenizer didn't 
lowercase the words before tokenizing, you would get "Alice" and "alice" appearing as two unique tokens. 
This would result in a word count of 9 unique tokens or words in the sentence out of 9 total unique words 
in this sample text vocabulary and likelihood of alice would be 1/9. If the other tokenizerdid lowercase all 
words before counting words, "Alice" and "alice" would appear as one token, which feels more accurate as they 
are referring to the same object, and you would get a word count of 8 unique tokens total in the sentence and 
likelihood 2/8. Prepocessing text by lowercasing everything first seems to be an important step in getting an 
accurate likelihood estimate for this sample text.
"""
)


A concrete example of two tokenizers giving different results on word count is if one tokenizer didn't 
lowercase the words before tokenizing, you would get "Alice" and "alice" appearing as two unique tokens. 
This would result in a word count of 9 unique tokens or words in the sentence out of 9 total unique words 
in this sample text vocabulary and likelihood of alice would be 1/9. If the other tokenizerdid lowercase all 
words before counting words, "Alice" and "alice" would appear as one token, which feels more accurate as they 
are referring to the same object, and you would get a word count of 8 unique tokens total in the sentence and 
likelihood 2/8. Prepocessing text by lowercasing everything first seems to be an important step in getting an 
accurate likelihood estimate for this sample text.



In [9]:
# q3b
### MULTIPLE CHOICE
### QUESTION: When tokenizing in this assignment we'll remove punctuation and discard numerical digits by making
#             everything lowercase and then capturing only consecutive letters a to z.
#             Suppose __`tokenize(x)`__ is a Python function that performs the desired tokenization.
#             What would __`tokenize("By-the-bye, what became of Alice's 12 hats?!")`__ output?

#   a.) ['by', 'the', 'bye', 'what', 'became', 'of', 'alice', 's', 'hats']
#   b.) ['by', 'the', 'bye', 'what', 'became', 'of', 'alices', '12', 'hats']
#   c.) ['by', 'the', 'bye', 'what', 'became', 'of', 'alices', 'hats']

### ENTER ONLY THE LETTER INSIDE THE PRINT STATEMENT.
# (i.e. if your answer is f.), it'll be:
#   answer = "f"

#####################
answer = "a"


#####################
print(answer)

a


In [10]:
# q3c
### CODE
### INSTRUCTIONS: Fill in the regular expression pattern in the cell labeled `part c` so that the subsequent
#                 call to `re.findall(RE_PATTERN, ...)` returns the tokenization described above.
#                 [`HINT`: _we've taken care of the lowercase part for you. If regex is new to you,
#                 check out the [`re`  documentation](https://docs.python.org/3/library/re.html) or
#                 [this PyMOTW tutorial](https://pymotw.com/2/re/)._]

### FILL IN YOUR ANSWER INSIDE THE: regex = " "
import re
def regex_tokenizer(text="By-the-bye, what became of Alice's 12 hats?!"):
    # BEGIN SOLUTION
    regex = "[a-zA-Z]+"
    # END SOLUTION

    RE_PATTERN = re.compile(regex)
    return re.findall(RE_PATTERN, text.lower())

# Load the Data
`"Please would you tell me", said Alice, a little timidly, for she was not quite sure whether it was good manners for her to speak first, "why your cat grins like that?"`  
<div style="text-align: right">  -- Lewis Carroll, <i>Alice's Adventures in Wonderland</i>, Chapter 4</div>

For the main part of this assignment we'll be working with the free plain text version of _Alice's Adventures in Wonderland_ available from Project Gutenberg. __Use the first two cells below to download this text and preview the first few lines.__ 

In [11]:
# Download Full text 
!mkdir -p data/
!gsutil cp gs://261-hw-data/main/Assignments/HW1/data/alice.txt data/alice.txt

Copying gs://261-hw-data/main/Assignments/HW1/data/alice.txt...
/ [1 files][166.6 KiB/166.6 KiB]                                                
Operation completed over 1 objects/166.6 KiB.                                    


In [12]:
# Take a peak at the first few lines:

# NOTE: If you are working in JupyterLab on Docker you may not see the output 
# below due to an encoding issue... in that case, use a terminal on Docker to 
# execute this head command and confirm that the file has downloaded properly, 
# this encoding issue should not affect your work on subsequent HW items.

!head -n 6 data/alice.txt

﻿The Project Gutenberg eBook of Alice’s Adventures in Wonderland, by Lewis Carroll

This eBook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You may copy it, give it away or re-use it under the terms
of the Project Gutenberg License included with this eBook or online at


We'd also like you to develop a habit of creating small files with simulated data for use in developing, debugging, and testing your code. The Jupyter magic command `%%writefile` is a convenient way to do this. __Run the following cells to create a test data file for use in our word counting task.__

In [13]:
%%writefile data/alice_test.txt
This is a small test file. This file is for a test.
This small test file has two small lines.

Overwriting data/alice_test.txt


In [14]:
# confirm the file was created in the data directory using a grep command:
!ls data | grep test

alice_test.txt


# Question 4: Word Count in Python

Over the course of this term you will also become very familiar with writing Python programs that read from standard input and using Linux piping commands to run these programs and save their output to file. __In this question you will write a short Python script to perform the Word Count task and then run your script on the _Alice in Wonderland_ text__. You can think of this like a "baseline implementation" that we'll later compare to the parallelized version of the same task.

### Q4 Tasks:

* __a) Code in `wordCount.py` and Submit on Gradescope:__ Complete the Python script in the file __`wordCount.py`__. Read the docstrings carefully to be sure you understand the expected behavior of this function. Please do not code outside of the marked location.


* __b) testing:__ Run the cell marked `part b` to call your script on the test file we created above. Confirm that your script returns the correct counts for each word by visually comparing the output to the test file. 


* __c) results:__ When you are confident in your implementation, run the cell marked `part c` to count the number of occurrences of each word in _Alice's Adventures in Wonderland_. In the same cell we'll pipe the output to file. Then use the provided `grep` commands to check your answers.


* __d) Short Essay:__ Suppose you decide that you'd really like  a word and its plural (e.g. 'hatter' and 'hatters' or 'person' and 'people') to be counted as the same word. After we have run the wordcount would it be more efficient to post-process your output file or discard your output file and start the analysis over with a new tokenizer? Explain your reasoning briefly. 

### Q4 Student Answers:
> __a-c)__ _Complete the coding portions of this question before answering 'd'._

In [15]:
# Make sure the files are in the correct encoding - RUN THIS CELL AS IS
!dos2unix wordCount.py

dos2unix: converting file wordCount.py to Unix format...


In [16]:
# part a - DO YOUR WORK IN wordCount.py

In [17]:
# RUN CELL as IS.
!cat wordCount.py

#!/usr/bin/env python
"""
This script reads lines from STDIN and returns a list of
all words and the count of how many times they occurred.

INPUT:
    a text file
OUTPUT FORMAT:
    word \t count
USAGE:
    python wordCount.py < yourTextFile.txt

Instructions:
    Fill in the missing code below so that the script
    prints tab separated word counts to Standard Output.
    NOTE: we have performed the tokenizing for you, please
    don't modify the provided code or you may fail unit tests.
"""

# imports
import sys
import re
from collections import defaultdict

counts = defaultdict(int)

# stream over lines from Standard Input
for line in sys.stdin:

    # tokenize
    line = line.strip()
    words = re.findall(r'[a-z]+', line.lower())

############ YOUR CODE HERE #########

    for word in words:
        counts[word] += 1
    
for word, count in counts.items():    
    print(f"{word}\t{count}")




############ (END) YOUR CODE #########


In [18]:
# part b - DO NOT MODIFY THIS CELL, just run it as is to test your script
!python wordCount.py < data/alice_test.txt

this	3
is	2
a	2
small	3
test	3
file	3
for	1
has	1
two	1
lines	1


In [19]:
# part c - DO NOT MODIFY THIS CELL, just run it as is to perform the word count.
!python wordCount.py < data/alice.txt > data/alice_counts.txt

Take a look at the first 10 words & their counts.

In [20]:
!head data/alice_counts.txt

the	1839
project	88
gutenberg	98
ebook	13
of	638
alice	403
s	222
adventures	11
in	435
wonderland	7


__Check your results:__ How many times does the word "alice" appear in the book? 

In [21]:
!grep alice data/alice_counts.txt

alice	403


In [22]:
# q4c1
### FILL IN
### QUESTION: How many times does the word "alice" appear in the book?

# ENTER YOUR ANSWER AS A VARIABLE USING THE 'num_alice_counts' variable below.

##############################
num_alice_counts = 403

##############################
# DON'T TOUCH
print(num_alice_counts)


403


__Check your results:__ How many times does the word "hatter" appear in the book? 

In [23]:
!grep hatter data/alice_counts.txt

hatter	56
hatters	1


In [24]:
# q4c2
### FILL IN
### QUESTION: How many times does the word "hatter" appear in the book?

# ENTER YOUR ANSWER AS A VARIABLE USING THE 'num_hatter_counts' variable below.

##############################
num_hatter_counts = 56

##############################
# DON'T TOUCH
print(num_hatter_counts)

56


__Check your results:__ How many times does the word "queen" appear in the book? 

In [25]:
!grep queen data/alice_counts.txt

queen	76
queens	1


In [26]:
# q4c3
### FILL IN
### QUESTION: How many times does the word "queen" appear in the book?

# ENTER YOUR ANSWER AS A VARIABLE USING THE 'num_queen_counts' variable below.

##############################
num_queen_counts = 76

##############################

# DON'T TOUCH
print(num_queen_counts)

76


In [27]:
# q4d
### SHORT RESPONSE
### QUESTION: Suppose you decide that you'd really like  a word and its plural (e.g. 'hatter' and 'hatters' or
#             'person' and 'people') to be counted as the same word. After we have run the wordcount would it
#             be more efficient to post-process your output file or discard your output file and start the
#             analysis over with a new tokenizer? Explain your reasoning briefly.

### ENTER ANSWER IN BETWEEN THE """ """ INSIDE THE PRINT STATEMENT.

print(
"""
It depends on which you need more of- time or memory. It would be more efficient in terms of time to create a 
new tokenizer that accounts for plural words- the operations preformed on the text tokens don't take too much 
time so building a new tokenizer takes less time. As for memory usage, it would take up more memory to create a
new dictionary with a new tokenizer. So it would depend on which is more important to us, which resource is more
scarce, time or memory. It takes more time to post-process but more memory to start the analysis over with a new
tokenizer. If we killed our cluster and started the whole process over, our CPU would "forget" the old
dictionary with old tokenizer and I believe it would be overall more 
efficient, in an entirely new run of the notebook, to create a new tokenizer and start the analysis over.
"""
)


It depends on which you need more of- time or memory. It would be more efficient in terms of time to create a 
new tokenizer that accounts for plural words- the operations preformed on the text tokens don't take too much 
time so building a new tokenizer takes less time. As for memory usage, it would take up more memory to create a
new dictionary with a new tokenizer. So it would depend on which is more important to us, which resource is more
scarce, time or memory. It takes more time to post-process but more memory to start the analysis over with a new
tokenizer. If we killed our cluster and started the whole process over, our CPU would "forget" the old
dictionary with old tokenizer and I believe it would be overall more 
efficient, in an entirely new run of the notebook, to create a new tokenizer and start the analysis over.



# Question 5: Unix Sorting Practice

Another common task in this course's assignments will be to make strategic use of sorting.     

### Q5 Tasks:
* __a) Multiple Choice:__ What is the Big O complexity of the fastest comparison based sorting algorithms? [*`HINT`: If you need a Big O notation refresher, here's a [blog post](https://rob-bell.net/2009/06/a-beginners-guide-to-big-o-notation/), a [cheatsheet](http://bigocheatsheet.com), and a [thorough explanation](http://pages.cs.wisc.edu/~vernon/cs367/notes/3.COMPLEXITY.html).*]

* __b) Multiple Choice:__ What is the default sorting algorithm in MapReduce? What is the Big O complexity of this algorithm? Why do you think this algorithm was chosen? [*`HINT`: Julius Ceasar! (week 1 slides)*]

* __c) Code in notebook:__ Write a unix command to check how many records are in your word count file. How many records are there?

* __d) Code in notebook:__ Write a unix command to sort your word count file alphabetically. Save (i.e. [redirect](https://superuser.com/questions/277324/pipes-vs-redirects)) the results to `data/alice_counts_A-Z.txt`. [*`HINT`: if Unix sort commands are new to you, start with [this biowize blogpost](https://biowize.wordpress.com/2015/03/13/unix-sort-sorting-with-both-numeric-and-non-numeric-keys/) or [this unixschool tutorial](http://www.theunixschool.com/2012/08/linux-sort-command-examples.html)*]

* __e) Code in notebook:__ Write a unix command to sort your word count file from highest to lowest count. Save (i.e. [redirect](https://superuser.com/questions/277324/pipes-vs-redirects)) your results to `data/alice_counts_sorted.txt`; then run the provided cell to print the top ten words. Compare your output to the expected output we provide.

### Q5 Student Answers:

In [28]:
# q5a
### MULTIPLE CHOICE
### QUESTION: What is the Big O complexity of the fastest comparison based sorting algorithms?
#            [*`HINT`: If you need a Big O notation refresher, here's a
#            [blog post](https://rob-bell.net/2009/06/a-beginners-guide-to-big-o-notation/),
#            a [cheatsheet](http://bigocheatsheet.com),
#            and a [thorough explanation](http://pages.cs.wisc.edu/~vernon/cs367/notes/3.COMPLEXITY.html).*]

#   a.) O(n log n)
#   b.) O(log n)
#   c.) O(n)
#   d.) O(2n)
#   e.) O(n!)

### ENTER ONLY THE LETTER INSIDE THE PRINT STATEMENT.
# (i.e. if your answer is f.), it'll be:
#   answer = "f"

### ENTER ONLY THE LETTER INSIDE THE PRINT STATEMENT. (i.e. if your answer is f.), enter 'f')
answer = "a"


#####################
print(answer)

a


In [29]:
# q5b
### MULTIPLE CHOICE
### QUESTION: What is the default sorting algorithm in MapReduce?
#             What is the Big O complexity of this algorithm? Why do you think this algorithm was chosen?
#             [*`HINT`: Julius Ceasar! (week 1 slides)*]

#   a.) Bubble sort. This is due to the simplicity of the algorithm.
#       The worst case time complexity of bubble sort is  𝑂(𝑛2)

#   b.) Quick sort. This is due to the partitioning nature of the algorithm
#       making it a perfect fit for parallelization. The worst case time complexity of bubble sort is  𝑂(𝑛⋅𝑙𝑜𝑔𝑛)

#   c.) Mergesort. This is due to the divide and conquer nature of the algorithm making it
#       a perfect fit for an embarrassingly parallel framework. The worst case time complexity of mergesort is  𝑂(𝑛⋅𝑙𝑜𝑔𝑛)

### ENTER ONLY THE LETTER INSIDE THE PRINT STATEMENT. (i.e. if your answer is f.), enter 'f')
answer = "c"


#####################
print(answer)

c


In [30]:
# part c - write a unix command to check how many records are in your word count file
#### WRITE BELOW
!wc -l data/alice_counts.txt

3006 data/alice_counts.txt


In [31]:
# q5c
### FILL IN
### QUESTION: Write a unix command to check how many records are in your word count file. How many records are there?

# ENTER YOUR ANSWER AS A VARIABLE USING THE 'num_records' variable below.

##############################
num_records = 3006

##############################

# DON'T TOUCH
print(num_records)

3006


In [32]:
# part d - unix command to sort your word counts alphabetically 
!sort data/alice_counts.txt > data/alice_counts_A-Z.txt

In [33]:
# part d - DO NOT MODIFY THIS CELL, run it as is to confirm your sort worked
!head data/alice_counts_A-Z.txt

a	695
abide	2
able	1
about	102
above	3
absence	1
absurd	2
accept	1
acceptance	1
accepted	2


In [34]:
# q5d
# DON'T MODIFY - Autograder Only
with open("data/alice_counts_A-Z.txt", "r") as f:
    for i in range(10):
        print(f.readline(), end="")

a	695
abide	2
able	1
about	102
above	3
absence	1
absurd	2
accept	1
acceptance	1
accepted	2


In [35]:
# part e - unix command to sort your word counts from highest to lowest count
!sort -k2,2nr data/alice_counts.txt > data/alice_counts_sorted.txt

In [36]:
# part e - DO NOT MODIFY THIS CELL, run it as is to confirm your sort worked
!head data/alice_counts_sorted.txt

the	1839
and	942
to	811
a	695
of	638
it	610
she	553
i	546
you	486
said	462


In [37]:
# q5e
# DON'T MODIFY - Autograder Only
with open("data/alice_counts_sorted.txt", "r") as f:
    for i in range(10):
        print(f.readline(), end="")

the	1839
and	942
to	811
a	695
of	638
it	610
she	553
i	546
you	486
said	462


<table>
<th>expected output for (d):</th>
<th>expected output for (e):</th>
<tr><td><pre>
a	695
abide	2
able	1
about	102
above	3
absence	1
absurd	2
accept	1
acceptance	1
accepted	2
</pre></td>
<td><pre>
the	1839
and	942
to	811
a	695
of	638
it	610
she	553
i	546
you	486
said	462
</pre></td></tr>
</table>

# Question 6: Parallel Word Count (part 1)
What would happen if we tried to run our script on a much larger dataset? For one thing, it would take longer to run. However there is a second concern. The Python object that aggregates our counts (`defaultdict`) exists in memory on the machine running this notebook. If the vocabulary is too large for the memory space available we would crash the notebook. The solution? Divide and Conquer! Instead of running the script on the whole dataset at once, we could split our text up in to smaller 'chunks' and process them independently of each other. __In this question you'll use a bash script to "parallelize" your Word Count.__


### Q6 Tasks:
* __a) Read provided code:__ The bash script `pWordCount_v1.sh` takes an input file, splits it into a specified number of 'chunks', and then applies an executable of your choice to each chunk. Read through this code and make sure you understand each line before you proceed. [*`HINT:` For now, ignore the 'student code' section -- you'll use that in part c.*]


* __b) Short Essay:__ Below we've provided the command to use this script to apply your analysis (`wordCount.py`) to the _Alice_ text in 4 parallel processes. We'll redirect the results into a file called `alice_pCounts.txt.` Run this analysis and compare the count for the word 'alice' to your answer from Question 4. Explain what went wrong and describe what we have to add to `pWordCount_v1.sh` to fix the problem.


* __c) Code in notebook:__ We've provided a python script, `aggregateCounts_v1.py`, which reads word counts from standard input and combines any duplicates it encounters. Read through this script to be sure you understand how it is written. Then follow the instructions in `pWordCount_v1.sh` to make a one-line modification so that it accepts `aggregateCounts_v1.py` as a 4th argument and uses this script to combine the chunk-ed word counts. Run the cell below to confirm that you now get the correct results for your 'alice' count.

### Q6 Student Answers:

In [38]:
# Make sure the files are in the correct encoding - RUN THIS CELL AS IS
!dos2unix pWordCount_v1.sh
!dos2unix aggregateCounts_v1.py

dos2unix: converting file pWordCount_v1.sh to Unix format...
dos2unix: converting file aggregateCounts_v1.py to Unix format...


In [39]:
# part b - RUN THIS CELL AS IS
!cat pWordCount_v1.sh

#!/bin/bash
# pWordCount.sh
# Author: James G. Shanahan
# Usage: pWordCount.sh m testFile.txt mapper.py [reducer.py]
# Input:
#   m = number of processes (maps), e.g., 4
#   inputFile = a text input file
#   mapper = an executable that reads from STDIN and prints to STDOUT
#   reducer = (optional) an executable that reads from STDIN and prints 
#             to STDOUT, if no reducer is provided, the framework will
#             simply stream the mapper output.
#
# Instructions:
#    For Q6a - Read this script and its comments closely. Ignore the
#              part marked "Otherwise" in STEP 3, you'll use that later.
#    For Q6c - Add a single line of code under '#Q6c' in STEP 3 so that
#              the script pipes the output of each chunk's word countfiles
#              into the second executable script provided as an argument,
#              Note that we saved the script name (which was the 4th arg)
#              to the variable $reducer. It can be executed by piping the
#     

In [40]:
# part b - make sure your scripts are executable (RUN THIS CELL AS IS)
!chmod a+x pWordCount_v1.sh
!chmod a+x wordCount.py

In [41]:
# part b - parallel word count on Alice text (RUN THIS CELL AS IS)
!./pWordCount_v1.sh 4 'data/alice.txt' 'wordCount.py' > 'data/alice_pCounts.txt'

In [42]:
# part b - check alice count (RUN THIS CELL AS IS)
!grep alice data/alice_pCounts.txt

alice	113
alice	126
alice	122
alice	42


In [43]:
# q6b
### SHORT RESPONSE
### QUESTION: Below we've provided the command to use this script to apply your analysis (`wordCount.py`)
#             to the _Alice_ text in 4 parallel processes. We'll redirect the results into a file called
#             `alice_pCounts.txt.` Run this analysis and compare the count for the word 'alice' to your
#             answer from Question 4. Explain what went wrong and describe what we have to add to
#             `pWordCount_v1.sh` to fix the problem.

### ENTER ANSWER IN BETWEEN THE """ """ INSIDE THE PRINT STATEMENT.
print(
"""
The currrent mapping process output 4 numbers that sum up to 403, the correct alice count, so to fix we need to add
the counts from the mapping process together, called reducing, a phase that comes after mapping. The mapping 
divided up the sum task but the current script didn't sum the mapped-sums (piece meal sums) together to output 
one final output, the total alice count. We need to add a reducer phase, a line of code so sum the total counts.
"""
)


The currrent mapping process output 4 numbers that sum up to 403, the correct alice count, so to fix we need to add
the counts from the mapping process together, called reducing, a phase that comes after mapping. The mapping 
divided up the sum task but the current script didn't sum the mapped-sums (piece meal sums) together to output 
one final output, the total alice count. We need to add a reducer phase, a line of code so sum the total counts.



In [44]:
# part c - make sure the aggregateCounts script is executable (RUN THIS CELL AS IS)
!chmod a+x aggregateCounts_v1.py

In [45]:
# part c - parallel word count on Alice text (RUN THIS CELL AS IS)
!./pWordCount_v1.sh 4 'data/alice.txt' \
                   'wordCount.py' \
                   'aggregateCounts_v1.py' \
                   > 'data/alice_pCounts.txt'

In [46]:
# part c - check alice count (RUN THIS CELL AS IS)
!grep alice data/alice_pCounts.txt

alice	403


# Question 7: Parallel Word Count (part 2)

Congratulations, you've just implemented a Map-Reduce algorithm! From here on out, we'll refer to the two Python scripts you passed to `pWordCount_v1.sh` as '_mapper_' and '_reducer_'. The bash script itself served as our '_framework_' -- it split up the original input, then ___mapped___ our word counting script on to each chunk, then ___aggregated (a.k.a. reduced)___ the resulting files by piping them into our collation script. Unfortunately, as you may have realized already, there is a major scalability concern with this particular implementation. __In this question you'll fix our implementation of parallel word count so that it will be scalable.__

__HINT:__ MapReduce uses the Merge-Sort algorithm under the hood. Linux `sort` command has a merge option which you can use to simulate the MapReduce framework. Use the `man sort` command to find more information on this option. 

### Q7 Tasks:

* __a) Multiple Choice:__ What is the potential scalability problem with the provided implementation of `aggregateCounts_v1.py`? Why would this supposedly 'parallelized' Word Count potentially not work on a really large input corpus. [*`HINT:` See the intro to Q6*]


* __b) Code in `pWordCount_v2.sh`:__ Fortunately, a 'strategic sort' can solve this problem. Read the instructions at the top of `pWordCount_v2.sh` carefully then make your changes that alphabetically sort the output from the mappers (`countfiles`) before piping them into the reducer script.


* __c) Code in `aggregateCounts_v2.py`:__ Write the main part of `aggregateCounts_v2.py` so that it takes advantage of the sorted input to add duplicate counts without storing the whole vocabulary in memory. Refer to the file docstring for more detailed instructions. Confirm that your implementation works by running it on both the test and true data files.


* __d) Short Essay:__ If you are paying close attention, this rewritten reducer sets us up for a truly scalable solution, but doesn't get us all the way there. In particular, while we chunked our data so it can be processed by multiple mappers, we're still streaming the entire dataset through one reduce script. If the vocabulary is too large to fit on a single computer, we might split the word counts in half after sorting them, then perform the reducing on two separate machines. Explain what could go wrong with this approach. (For now, ignore the question of how we'd sort a dataset that is too large to fit on a single machine and just focus on what might be wrong about the result of this split-in-half reducing).


* __e) Short Essay:__ Can you come up with a different way of splitting up the data that would allow us to perform the reducing on separate machines without needing any postprocessing? This is a theoretical question -- don't worry if you don't know how to implement your idea in a bash script, just describe how you'd want to split the sorted counts into different files to be reduced separately.

### Q7 Student Answers:

In [47]:
# Make sure the files are in the correct encoding - RUN THIS CELL AS IS
!dos2unix pWordCount_v2.sh
!dos2unix aggregateCounts_v2.py

dos2unix: converting file pWordCount_v2.sh to Unix format...
dos2unix: converting file aggregateCounts_v2.py to Unix format...


In [48]:
# q7a
### MULTIPLE CHOICE
### QUESTION: What is the potential scalability problem with the provided implementation of `aggregateCounts_v1.py`?
#             Why would this supposedly 'parallelized' Word Count potentially not work on a really large
#             input corpus. [*`HINT:` See the intro to Q6*]

#   a.) The implementation stores every line from sys.stdin in memory while accumulating counts.
#       For a really large corpus and/or a cluster of machines with memory constraints
#       this could become too large to run on a single node.

#   b.) The implementation requires Python imports which may not be available on every
#       node of the cluster for the mappers to use.

#   c.) The implementation stores the entire vocabulary in a dictionary while accumulating counts.
#       For a really large corpus and/or a cluster of machines with memory constraints this dictionary 
#       could become too large to run on a single node. In fact, from a scalability perspective this
#       implementation is essentially equivalent to our original python word counter.

### ENTER ONLY THE LETTER INSIDE THE PRINT STATEMENT. (i.e. if your answer is f.), enter 'f')
answer = "c"


#####################
print(answer)

c


In [98]:
# Run CELL AS IS
!cat pWordCount_v2.sh

#!/bin/bash
# pWordCount.sh
# Author: James G. Shanahan
# Usage: pWordCount.sh m testFile.txt mapper.py [reducer.py]
# Input:
#   m = number of processes (maps), e.g., 4
#   inputFile = a text input file
#   mapper = an executable that reads from STDIN and prints to STDOUT
#   reducer = (optional) an executable that reads from STDIN and prints 
#             to STDOUT, if no reducer is provided, the framework will
#             simply stream the mapper output.
#
# Instructions:
#    For Q7b - Ammend this script in STEP 2 and STEP 3 to 
#              alphabtetically sort the contents of each chunk before
#              piping them into the reducer script and redirecting on to 
# .            $data.output.
# --------------------------------------------------------------------

usage()
{
    echo ERROR: No arguments supplied
    echo
    echo To run use
    echo "pWordCount.sh m inputFile mapper.py [reducer.py]"
    echo Input:
    echo "number of processes/maps, EG, 4"
    echo "mapper.

In [85]:
# Run CELL AS IS
!cat aggregateCounts_v2.py

#!/usr/bin/env python
"""
This script reads word counts from STDIN and aggregates
the counts for any duplicated words.

INPUT & OUTPUT FORMAT:
    word \t count
USAGE (standalone):
    python aggregateCounts_v2.py < yourCountsFile.txt

Instructions:
    For Q7 - Your solution should not use a dictionary or store anything   
             other than a single total count - just print them as soon as  
             you've added them. HINT: you've modified the framework script 
             to ensure that the input is alphabetized; how can you 
             use that to your advantage?
"""

# imports
import sys


################# YOUR CODE HERE #################
# initialize trackers
cur_word = None
cur_count = 0

# read input key-value pairs from standard input
for line in sys.stdin:
    key, value = line.split()
    # tally counts from current key
    if key == cur_word: 
        cur_count += int(value)
    # OR emit current total and start a new tally 
    else: 
        if cur_word:
   

In [86]:
# Add permissions to your new files (RUN THIS CELL AS IS)
!chmod a+x pWordCount_v2.sh
!chmod a+x aggregateCounts_v2.py

In [95]:
# part c - test your code on the test file (RUN THIS CELL AS IS)
!./pWordCount_v2.sh 4 'data/alice_test.txt' \
                   'wordCount.py' \
                   'aggregateCounts_v2.py'

a	2
file	3
for	1
has	1
is	2
lines	1
small	3
test	3
this	3
two	1


In [96]:
# part c - run your code on the Alice file (RUN THIS CELL AS IS)
!./pWordCount_v2.sh 4 'data/alice.txt' \
                   'wordCount.py' \
                   'aggregateCounts_v2.py' \
                   > 'data/alice_pCounts.txt'

In [97]:
# part c - confirm that your 'alice' count is correct (RUN THIS CELL AS IS)
!grep alice data/alice_pCounts.txt

alice	403


In [99]:
# q7d
### SHORT RESPONSE
### QUESTION: If you are paying close attention, this rewritten reducer sets us up for a truly scalable
#             solution, but doesn't get us all the way there. In particular, while we chunked our data
#             so it can be processed by multiple mappers, we're still streaming the entire dataset
#             through one reduce script. If the vocabulary is too large to fit on a single computer, we
#             might split the word counts in half after sorting them, then perform the reducing on two
#             separate machines. Explain what could go wrong with this approach. (For now, ignore the
#             question of how we'd sort a dataset that is too large to fit on a single machine and
#             just focus on what might be wrong about the result of this split-in-half reducing).

### ENTER ANSWER IN BETWEEN THE """ """ INSIDE THE PRINT STATEMENT.

print(
"""
Given the current implementation, we wouldn't be able to sort the whole dataset to aggregate (reduce) counts. I 
had to sort twice, once with the full dataset, and if we couldn't do that on one machine our reducer wouldn't get 
a dataset with all sorted keys and so it wouldn't be able to aggregate counts, we would have duplicated keys with 
split counts.
"""
)


Given the current implementation, we wouldn't be able to sort the whole dataset to aggregate (reduce) counts. I 
had to sort twice, once with the full dataset, and if we couldn't do that on one machine our reducer wouldn't get 
a dataset with all sorted keys and so it wouldn't be able to aggregate counts, we would have duplicated keys with 
split counts.



In [100]:
# q7e
### SHORT RESPONSE
### QUESTION: Can you come up with a different way of splitting up the data that would allow us to perform
#             the reducing on separate machines without needing any postprocessing? This is a theoretical
#             question -- don't worry if you don't know how to implement your idea in a bash script,
#             just describe how you'd want to split the sorted counts into different files to be reduced separately.

### ENTER ANSWER IN BETWEEN THE """ """ INSIDE THE PRINT STATEMENT.

print(
"""
You could sort the split sorted counts again so that when you reduce separately you are working with sequential
data so you don't have to worry about aggregating- it's already been split based on sequential sort. Add another 
sort layer so when you reduce, each key is uniquely split and you don't need to worry about matching keys across
different files.
"""
)


You could sort the split sorted counts again so that when you reduce separately you are working with sequential
data so you don't have to worry about aggregating- it's already been split based on sequential sort. Add another 
sort layer so when you reduce, each key is uniquely split and you don't need to worry about matching keys across
different files.



# Question 8: Scalability Considerations

In your reading for Week 2's live session, [Chapter1, section 2](https://lintool.github.io/MapReduceAlgorithms/MapReduce-book-final.pdf) of _Data Intensive Text Processing with MapReduce_, Lin and Dyer discuss a number of "Big Ideas" that underlie large scale processing: __scale "out," not "up"; assume failures are common; move processing to the data; process data sequentially and avoid random access; hide system-level details from the application developer; and seamless scalability.__ Part of our work this semester will be to interact with these ideas in a practical way, not just a conceptual one.

### Q8 Tasks:

* __a) Short Essay:__ What do Lin and Dyer consider the two features of an "ideal algorithm" from a scalability perspective?


* __b) Multiple Choice:__ The mapper script below (created on the fly using a little Jupyter magic) will help us illustrate the concept of scalability. Run the provided code which passes this mapper script to our parallel computation 'framework' and runs the 'analysis' on the _Alice_ text file. Note that we've omitted a reducer for simplicity. What do you observe about the time it takes to run this "algorithm" when we use 1, 2 and 4 partitions? Does it meet Lin and Dyer's criteria?

* __c) Short Essay:__ Let's try something similar with your Word Count analysis. Run the provided code to time your implementation with 1, 2, 4 and 8 partitions. What do you observe about the runtimes? Does this match your expectation? Speculate about why we might be seeing these times. What conclusions should we draw about the scalability of our implementation? [*`HINT:` Consider the limitations of both your machine and our implementation... there are some competing forces at work, what are they?*]


* __d) Multiple Choice:__ Which components of your Map-Reduce algorithm are affected by a change in the number of partitions? Does increasing the number of partitions increase or decrease the total time spent on each of these portions of the task? What tradeoff does this cause?

### Q8 Student Answers:

In [101]:
# q8a
### SHORT RESPONSE
### QUESTION: What do Lin and Dyer consider the two features of an "ideal algorithm" from a scalability perspective?

### ENTER ANSWER IN BETWEEN THE """ """ INSIDE THE PRINT STATEMENT.

print(
"""
According to the text book, an "ideal algorithm" will have scalability across 2 dimensions: given twice the 
amount of data the algorithm should at most ttake twice as long to run, and second, given a cluster twice the 
size, the same algorithm shoudl take no more than half as long to run. These things would happen with no
modifications necessary depending on the size and type of data. So essentially you can manage time it takes to 
run an algorithm by increasing clusters proportional to the increase in amount of data.
"""
)


According to the text book, an "ideal algorithm" will have scalability across 2 dimensions: given twice the 
amount of data the algorithm should at most ttake twice as long to run, and second, given a cluster twice the 
size, the same algorithm shoudl take no more than half as long to run. These things would happen with no
modifications necessary depending on the size and type of data. So essentially you can manage time it takes to 
run an algorithm by increasing clusters proportional to the increase in amount of data.



__Run the following cells to create the mapper referenced in `part b`__

In [102]:
!mkdir demo

mkdir: cannot create directory ‘demo’: File exists


In [103]:
%%writefile demo/mapper.py
#!/usr/bin/env python
"""
This mapper reads from STDIN and waits 0.001 seconds per line.
Its only purpose is to demonstrate one of the scalability ideas.
"""
import sys
import time
for line in sys.stdin:
    time.sleep(0.001)

Overwriting demo/mapper.py


In [104]:
# Make sure the mapper is executable
!chmod a+x demo/mapper.py

__Run the next three cells to apply our demo mapper with 1, 2 and 4 partitions.__

In [105]:
%%timeit
!./pWordCount_v2.sh 1 'data/alice.txt' 'demo/mapper.py'

4.44 s ± 53.8 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [106]:
%%timeit
!./pWordCount_v2.sh 2 'data/alice.txt' 'demo/mapper.py'

2.32 s ± 11.8 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [107]:
%%timeit
!./pWordCount_v2.sh 4 'data/alice.txt' 'demo/mapper.py'

1.3 s ± 4.12 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


__Run the following cells to repeat this process with your word count algorithm refereced in `part c`.__

In [108]:
%%timeit
!./pWordCount_v2.sh 1 'data/alice.txt' 'wordCount.py' 'aggregateCounts_v2.py' > 'data/tmp'

314 ms ± 1.79 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [109]:
%%timeit
!./pWordCount_v2.sh 2 'data/alice.txt' 'wordCount.py' 'aggregateCounts_v2.py' > 'data/tmp'

330 ms ± 9.41 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [110]:
%%timeit
!./pWordCount_v2.sh 4 'data/alice.txt' 'wordCount.py' 'aggregateCounts_v2.py' > 'data/tmp'

343 ms ± 11.4 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [111]:
%%timeit
!./pWordCount_v2.sh 8 'data/alice.txt' 'wordCount.py' 'aggregateCounts_v2.py' > 'data/tmp'

440 ms ± 7.03 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [112]:
%%timeit
!./pWordCount_v2.sh 16 'data/alice.txt' 'wordCount.py' 'aggregateCounts_v2.py' > 'data/tmp'

677 ms ± 14.1 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [113]:
%%timeit
!./pWordCount_v2.sh 32 'data/alice.txt' 'wordCount.py' 'aggregateCounts_v2.py' > 'data/tmp'

1.02 s ± 17.1 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [114]:
# q8b
### MULTIPLE CHOICE
### QUESTION: The mapper script below (created on the fly using a little Jupyter magic) will help us illustrate
#             the concept of scalability. Run the provided code which passes this mapper script to our parallel
#             computation 'framework' and runs the 'analysis' on the _Alice_ text file. Note that we've omitted a reducer for simplicity. What do you observe about the time it takes to run this "algorithm" when we use 1, 2 and 4 partitions? Does it meet Lin and Dyer's criteria?

#   a.) Yes, doubling the number of map tasks (approximately) halves the runtime.
#   b.) No, doubling the number of map tasks the runtime stays the same or even increases.

### ENTER ONLY THE LETTER INSIDE THE PRINT STATEMENT. (i.e. if your answer is f.), enter 'f')
answer = "a"


#####################
print(answer)

a


In [115]:
# q8c
### SHORT RESPONSE
### QUESTION: Let's try something similar with your Word Count analysis. Run the provided code to time your
#             implementation with 1, 2, 4 and 8 partitions. What do you observe about the runtimes? Does this
#             match your expectation? Speculate about why we might be seeing these times. What conclusions
#             should we draw about the scalability of our implementation? [*`HINT:` Consider the limitations
#             of both your machine and our implementation... there are some competing forces at work, what are they?*]

### ENTER ANSWER IN BETWEEN THE """ """ INSIDE THE PRINT STATEMENT.

print(
"""
The run times are not reducing as you increase partitions, they are slightly increasing. This might be happening 
because we are sorting the whole dataset so the whole alice text is being run through my computers memory
even though we aren't using a dictionary, we are still sorting in the aggregation (reduce) phase of the 
algorithim which takes more time as the partitions and aggregation steps increase.
"""
)


The run times are not reducing as you increase partitions, they are slightly increasing. This might be happening 
because we are sorting the whole dataset so the whole alice text is being run through my computers memory
even though we aren't using a dictionary, we are still sorting in the aggregation (reduce) phase of the 
algorithim which takes more time as the partitions and aggregation steps increase.



In [116]:
# q8d
### MULTIPLE CHOICE
### QUESTION: Which components of your Map-Reduce algorithm are affected by a change in the number of partitions?
#             Does increasing the number of partitions increase or decrease the total time spent on each of these
#             portions of the task? What tradeoff does this cause?

#   a.) The map/reduce phases will be shorter with more partitions but the sort will take longer because
#       there will be more records to shuffle.

#   b.) The map/reduce phases will be longer with more partitions but the sort will be faster because
#       the records are more distributed.

### ENTER ONLY THE LETTER INSIDE THE PRINT STATEMENT. (i.e. if your answer is f.), enter 'f')
answer = "a"


#####################
print(answer)

a


# Question 9: Embarrassingly Parallel
`"If any one of them can explain it," said Alice, (she had grown so large in the last few minutes that she wasn’t a bit afraid of interrupting him,) "I’ll give him sixpence. I don’t believe there’s an atom of meaning in it."`
<div style="text-align: right">  -- Lewis Carroll, <i>Alice's Adventures in Wonderland</i>, Chapter 12</div>

### Q9 Tasks:

* __a) Short Essay:__ Describe what we mean by 'Embarrassingly Parallel' in terms of word counting. Does this term describe a 'task'? An 'implementation of a task'? 

* __b) Short Essay:__ Define this concept in terms of 'associative' and 'commutative' operations. [*`HINT:` Refer to Chapter 2 in DITP*]

In [117]:
# q9a
### SHORT RESPONSE
### QUESTION: Describe what we mean by 'Embarrassingly Parallel' in terms of word counting.
#             Does this term describe a 'task'? An 'implementation of a task'? 

### ENTER ANSWER IN BETWEEN THE """ """ INSIDE THE PRINT STATEMENT.

print(
"""
This term describes an implementation of a task - counting is an incredibly redundant process so doing it in 
parallel increases efficiency of the task of counting features of a large data set. 
"""
)


This term describes an implementation of a task - counting is an incredibly redundant process so doing it in 
parallel increases efficiency of the task of counting features of a large data set. 



In [118]:
# q9b
### SHORT RESPONSE
### QUESTION: Define this concept in terms of 'associative' and 'commutative' operations.
#             [*`HINT:` Refer to Chapter 2 in DITP*]

### ENTER ANSWER IN BETWEEN THE """ """ INSIDE THE PRINT STATEMENT.

print(
"""
Assoicative and communative operations are related to functional programing roots. Mapping is an associative 
operation, like counting in our example in this homework, as it takes as an argument a function, counting, and 
applies it to all elements of a list, words in our dataset. Folding, or reducing, takes the result of the function 
that takes two arguments, in our example words as keys and their counts as values, and iterates through that
process of taking two arguments in a function and storing them temporarily, finally outputting the final variable
when all items in list (words as keys in our dataset) have been consumed. Fold is the aggregation or communtative
process, mapping is an associative process. MapReduce is designed to increase efficiency by adding reordering and 
aggregation in the fold operation or reduce or commutative phase of these operations, essentially.
"""
)


Assoicative and communative operations are related to functional programing roots. Mapping is an associative 
operation, like counting in our example in this homework, as it takes as an argument a function, counting, and 
applies it to all elements of a list, words in our dataset. Folding, or reducing, takes the result of the function 
that takes two arguments, in our example words as keys and their counts as values, and iterates through that
process of taking two arguments in a function and storing them temporarily, finally outputting the final variable
when all items in list (words as keys in our dataset) have been consumed. Fold is the aggregation or communtative
process, mapping is an associative process. MapReduce is designed to increase efficiency by adding reordering and 
aggregation in the fold operation or reduce or commutative phase of these operations, essentially.



### Congratulations, you have completed HW1! Please submit this notebook (HW1.ipynb) and the following files.
#### Please make sure that the filenames exactly match:
- HW1.ipynb
- aggregateCounts_v1.py
- aggregateCounts_v2.py
- pWordCount_v1.sh
- pWordCount_v2.sh
- wordCount.py
- alice_counts_A-Z.txt
- alice_counts_sorted.txt